# Salary Prediction — Full Notebook
### Phase 1: Cleaning + EDA  |  Phase 2: Training + Tuning

In [ ]:
# Install exact versions for reproducibility
!pip install scikit-learn==1.4.2 pandas==2.2.2 joblib==1.4.2 seaborn==0.13.2 matplotlib==3.8.4 --quiet
print('All packages installed.')

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
print('All imports loaded successfully.')

## Section 1 — Load & Explore

In [ ]:
df = pd.read_csv('/content/ds_salaries.csv')

print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns\n')
print('Column names:')
print(df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('=' * 55)
print('DATA TYPES')
print('=' * 55)
print(df.dtypes)

print('\n' + '=' * 55)
print('UNIQUE VALUE COUNTS PER COLUMN')
print('=' * 55)
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique values')

print('\n' + '=' * 55)
print('SAMPLE VALUES FOR CATEGORICAL COLUMNS')
print('=' * 55)
for col in ['experience_level', 'employment_type', 'company_size', 'remote_ratio']:
    print(f'\n  {col}: {df[col].unique().tolist()}')

In [ ]:
print('=' * 55)
print('NULL VALUES PER COLUMN')
print('=' * 55)
null_counts = df.isnull().sum()
print(null_counts)
print(f'\nTotal null values: {null_counts.sum()}')

print('\n' + '=' * 55)
print('DUPLICATE ROWS')
print('=' * 55)
duplicate_count = df.duplicated().sum()
print(f'Duplicate rows: {duplicate_count}')

if duplicate_count > 0:
    df = df.drop_duplicates(keep='first')
    print(f'Dropped {duplicate_count} duplicate rows. New shape: {df.shape}')
else:
    print('No duplicates found.')

## Section 2 — EDA Visualizations
> Run before encoding so labels are human-readable.

In [ ]:
# Global chart style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.facecolor':   'white',
    'axes.facecolor':     '#f9f9f9',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.titlesize':     14,
    'axes.titleweight':   'bold',
    'axes.titlepad':      14,
    'figure.dpi':         130,
})

EXPERIENCE_LABELS = {'EN': 'Entry', 'MI': 'Mid', 'SE': 'Senior', 'EX': 'Executive'}
EMPLOYMENT_LABELS = {'FT': 'Full-time', 'PT': 'Part-time', 'CT': 'Contract', 'FL': 'Freelance'}
SIZE_LABELS       = {'S': 'Small', 'M': 'Medium', 'L': 'Large'}

df_plot = df.copy()
df_plot['experience_level'] = df_plot['experience_level'].map(EXPERIENCE_LABELS)
df_plot['employment_type']  = df_plot['employment_type'].map(EMPLOYMENT_LABELS)
df_plot['company_size']     = df_plot['company_size'].map(SIZE_LABELS)

print(f'Plot style ready. Working with {len(df_plot)} rows.')

In [ ]:
# Chart 1: Salary Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Salary Distribution (USD)', fontsize=16, fontweight='bold', y=1.02)

sns.histplot(data=df_plot, x='salary_in_usd', bins=40, kde=True,
             color='#4C72B0', ax=axes[0])
axes[0].set_title('Full Distribution')
axes[0].set_xlabel('Annual Salary (USD)')
axes[0].set_ylabel('Number of Employees')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

median_sal = df_plot['salary_in_usd'].median()
mean_sal   = df_plot['salary_in_usd'].mean()
axes[0].axvline(median_sal, color='#DD8452', linestyle='--', linewidth=1.5,
                label=f'Median  ${median_sal/1000:.0f}k')
axes[0].axvline(mean_sal,   color='#55A868', linestyle='--', linewidth=1.5,
                label=f'Mean    ${mean_sal/1000:.0f}k')
axes[0].legend(fontsize=10)

sns.boxplot(data=df_plot, y='salary_in_usd', color='#4C72B0', width=0.4,
            flierprops={'marker': 'o', 'markerfacecolor': '#DD8452',
                        'markersize': 4, 'alpha': 0.5}, ax=axes[1])
axes[1].set_title('Spread & Outliers')
axes[1].set_ylabel('Annual Salary (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1000:.0f}k'))

plt.tight_layout()
plt.savefig('/content/chart_01_salary_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Median: ${median_sal:,.0f} | Mean: ${mean_sal:,.0f}')

In [ ]:
# Chart 2: Salary by Experience Level
EXPERIENCE_ORDER = ['Entry', 'Mid', 'Senior', 'Executive']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Salary by Experience Level', fontsize=16, fontweight='bold')

sns.violinplot(data=df_plot, x='experience_level', y='salary_in_usd',
               order=EXPERIENCE_ORDER, palette='muted', inner='quartile', ax=axes[0])
axes[0].set_title('Distribution Shape')
axes[0].set_xlabel('Experience Level')
axes[0].set_ylabel('Annual Salary (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1000:.0f}k'))

mean_by_exp = (df_plot.groupby('experience_level')['salary_in_usd']
               .mean().reindex(EXPERIENCE_ORDER).reset_index())
bars = axes[1].bar(mean_by_exp['experience_level'], mean_by_exp['salary_in_usd'],
                   color=sns.color_palette('muted', 4), edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, mean_by_exp['salary_in_usd']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
                 f'${val/1000:.0f}k', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_title('Mean Salary')
axes[1].set_xlabel('Experience Level')
axes[1].set_ylabel('Mean Annual Salary (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1000:.0f}k'))
axes[1].set_ylim(0, mean_by_exp['salary_in_usd'].max() * 1.2)

plt.tight_layout()
plt.savefig('/content/chart_02_salary_by_experience.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Chart 3: Top 15 Paying Job Titles
top_n = 15
top_titles = (df_plot.groupby('job_title')['salary_in_usd']
              .agg(median_salary='median', count='count')
              .query('count >= 5')
              .sort_values('median_salary', ascending=True)
              .tail(top_n).reset_index())

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_titles['job_title'], top_titles['median_salary'],
               color=sns.color_palette('Blues_d', top_n), edgecolor='white', linewidth=0.6)
for bar, val in zip(bars, top_titles['median_salary']):
    ax.text(val + 1000, bar.get_y() + bar.get_height()/2,
            f'${val/1000:.0f}k', va='center', fontsize=10)
ax.set_title(f'Top {top_n} Job Titles by Median Salary (5+ data points)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Median Annual Salary (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.set_xlim(0, top_titles['median_salary'].max() * 1.18)
plt.tight_layout()
plt.savefig('/content/chart_03_top_job_titles.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Chart 4: Remote Work vs Salary
REMOTE_LABELS = {0: 'On-site\n(0%)', 50: 'Hybrid\n(50%)', 100: 'Fully remote\n(100%)'}
df_plot['remote_label'] = df_plot['remote_ratio'].map(REMOTE_LABELS)
REMOTE_ORDER = ['On-site\n(0%)', 'Hybrid\n(50%)', 'Fully remote\n(100%)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Remote Work vs. Salary', fontsize=16, fontweight='bold')

remote_counts = df_plot['remote_label'].value_counts().reindex(REMOTE_ORDER)
axes[0].bar(REMOTE_ORDER, remote_counts.values,
            color=sns.color_palette('Set2', 3), edgecolor='white')
for i, v in enumerate(remote_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')
axes[0].set_title('Headcount per Work Type')
axes[0].set_ylabel('Number of Employees')

remote_salary = df_plot.groupby('remote_label')['salary_in_usd'].median().reindex(REMOTE_ORDER)
bars = axes[1].bar(REMOTE_ORDER, remote_salary.values,
                   color=sns.color_palette('Set2', 3), edgecolor='white')
for bar, val in zip(bars, remote_salary.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'${val/1000:.0f}k', ha='center', fontweight='bold')
axes[1].set_title('Median Salary by Work Type')
axes[1].set_ylabel('Median Annual Salary (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1000:.0f}k'))
axes[1].set_ylim(0, remote_salary.max() * 1.2)

plt.tight_layout()
plt.savefig('/content/chart_04_remote_vs_salary.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Chart 5: Salary Trend Over Years
yearly = (df_plot.groupby('work_year')['salary_in_usd']
          .agg(median='median',
               q25=lambda x: x.quantile(0.25),
               q75=lambda x: x.quantile(0.75),
               count='count').reset_index())

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(yearly['work_year'], yearly['q25'], yearly['q75'],
                alpha=0.2, color='#4C72B0', label='25th-75th percentile')
ax.plot(yearly['work_year'], yearly['median'],
        marker='o', markersize=8, linewidth=2.5, color='#4C72B0', label='Median salary')
for _, row in yearly.iterrows():
    ax.annotate(f'${row["median"]/1000:.0f}k\n(n={int(row["count"])})',
                xy=(row['work_year'], row['median']),
                xytext=(0, 14), textcoords='offset points',
                ha='center', fontsize=9)
ax.set_title('Median Data Science Salary Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Annual Salary (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1000:.0f}k'))
ax.set_xticks(yearly['work_year'])
ax.legend()
plt.tight_layout()
plt.savefig('/content/chart_05_salary_trend.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Chart 6: Heatmap — Experience x Company Size
SIZE_ORDER       = ['Small', 'Medium', 'Large']
EXPERIENCE_ORDER = ['Entry', 'Mid', 'Senior', 'Executive']

pivot = (df_plot.groupby(['experience_level', 'company_size'])['salary_in_usd']
         .median().unstack('company_size')
         .reindex(index=EXPERIENCE_ORDER, columns=SIZE_ORDER))

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot / 1000, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Median Salary ($ thousands)'}, ax=ax)
ax.set_title('Median Salary (k USD)\nExperience Level x Company Size',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Company Size')
ax.set_ylabel('Experience Level')
plt.tight_layout()
plt.savefig('/content/chart_06_heatmap_exp_size.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
from google.colab import files

eda_charts = [
    '/content/chart_01_salary_distribution.png',
    '/content/chart_02_salary_by_experience.png',
    '/content/chart_03_top_job_titles.png',
    '/content/chart_04_remote_vs_salary.png',
    '/content/chart_05_salary_trend.png',
    '/content/chart_06_heatmap_exp_size.png',
]
for path in eda_charts:
    files.download(path)
    print(f'Downloaded: {path.split("/")[-1]}')
print('\nAll EDA charts downloaded.')

## Section 3 — Cleaning & Encoding

In [ ]:
# Drop redundant salary columns — we only need salary_in_usd
COLUMNS_TO_DROP = ['salary', 'salary_currency']
df = df.drop(columns=COLUMNS_TO_DROP)
print(f'Dropped: {COLUMNS_TO_DROP}')
print(f'Remaining columns: {df.columns.tolist()}')

In [ ]:
# Group rare job titles (< 10 occurrences) into 'Other'
# Prevents the model from overfitting on titles it barely sees
MIN_OCCURRENCES = 10
title_counts = df['job_title'].value_counts()
rare_titles  = title_counts[title_counts < MIN_OCCURRENCES].index.tolist()

print(f'Unique job titles before: {df["job_title"].nunique()}')
print(f'Titles being grouped into Other ({len(rare_titles)}): {rare_titles}')

df['job_title'] = df['job_title'].apply(
    lambda t: t if t not in rare_titles else 'Other'
)
print(f'Unique job titles after : {df["job_title"].nunique()}')

In [ ]:
# Encode all categorical columns to integers
# We save each encoder — the API needs them to convert string inputs
CATEGORICAL_COLUMNS = [
    'experience_level', 'employment_type', 'job_title',
    'employee_residence', 'company_location', 'company_size',
]

encoders = {}
print('Encoding categorical columns...\n')

for col in CATEGORICAL_COLUMNS:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col])
    encoders[col] = encoder
    print(f'  {col}: {len(encoder.classes_)} classes -> integers 0 to {len(encoder.classes_)-1}')

print('\nEncoding complete.')

In [ ]:
# Final sanity check before saving
print('Null values remaining:', df.isnull().sum().sum())
print('\nData types:')
print(df.dtypes)
print('\nTarget stats (salary_in_usd):')
print(df['salary_in_usd'].describe().round(2))

In [ ]:
CLEAN_CSV_PATH = '/content/ds_salaries_clean.csv'
ENCODERS_PATH  = '/content/encoders.pkl'

df.to_csv(CLEAN_CSV_PATH, index=False)
joblib.dump(encoders, ENCODERS_PATH)

print(f'Clean dataset saved -> {CLEAN_CSV_PATH}  shape: {df.shape}')
print(f'Encoders saved      -> {ENCODERS_PATH}   keys: {list(encoders.keys())}')

In [ ]:
files.download(CLEAN_CSV_PATH)
files.download(ENCODERS_PATH)
print('Downloaded: ds_salaries_clean.csv + encoders.pkl')

## Section 4 — Model Training & Tuning

In [ ]:
# ── Outlier removal ──
# Extreme salary values inflate RMSE and mislead the model.
# We use the IQR fence method: anything beyond 1.5x the IQR
# above Q3 or below Q1 is considered an outlier.
q1  = df['salary_in_usd'].quantile(0.25)
q3  = df['salary_in_usd'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

df_model = df[
    (df['salary_in_usd'] >= lower_fence) &
    (df['salary_in_usd'] <= upper_fence)
].copy()

print(f'Rows before outlier removal : {len(df)}')
print(f'Rows after  outlier removal : {len(df_model)}')
print(f'Removed                     : {len(df) - len(df_model)} outlier rows')
print(f'Salary range after removal  : ${df_model["salary_in_usd"].min():,.0f} - ${df_model["salary_in_usd"].max():,.0f}')

In [ ]:
# ── Features and target ──
# work_year IS included — salaries changed significantly year over year
TARGET_COL   = 'salary_in_usd'
FEATURE_COLS = [col for col in df_model.columns if col != TARGET_COL]

X = df_model[FEATURE_COLS]
y = df_model[TARGET_COL]

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Target: {TARGET_COL}')
print(f'X shape: {X.shape} | y range: ${y.min():,.0f} - ${y.max():,.0f}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training set : {len(X_train)} rows ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test set     : {len(X_test)} rows ({len(X_test)/len(X)*100:.1f}%)')

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name):
    """
    Train a model and report MAE, RMSE, R² on train + test + 5-fold CV.

    Args:
        model:      unfitted sklearn estimator
        X_tr:       training features
        X_te:       test features
        y_tr:       training target
        y_te:       test target
        model_name: label for printing

    Returns:
        dict: metrics and fitted model
    """
    model.fit(X_tr, y_tr)
    y_pred_test  = model.predict(X_te)
    y_pred_train = model.predict(X_tr)

    mae      = mean_absolute_error(y_te, y_pred_test)
    rmse     = np.sqrt(mean_squared_error(y_te, y_pred_test))
    r2       = r2_score(y_te, y_pred_test)
    r2_train = r2_score(y_tr, y_pred_train)
    cv_r2    = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()

    print(f'\n{"="*50}')
    print(f'  {model_name}')
    print(f'{"="*50}')
    print(f'  MAE            : ${mae:>10,.0f}')
    print(f'  RMSE           : ${rmse:>10,.0f}')
    print(f'  R2 (test)      : {r2:>10.4f}')
    print(f'  R2 (train)     : {r2_train:>10.4f}')
    print(f'  R2 (5-fold cv) : {cv_r2:>10.4f}')

    gap = r2_train - r2
    if gap > 0.15:
        print(f'\n  WARNING: gap {gap:.2f} — possible overfit')
    else:
        print(f'\n  OK: train/test gap {gap:.2f} — model generalizes well')

    return {'name': model_name, 'model': model, 'mae': mae, 'rmse': rmse,
            'r2_test': r2, 'r2_train': r2_train, 'r2_cv': cv_r2, 'y_pred': y_pred_test}

print('evaluate_model() ready.')

In [ ]:
# Find best max_depth for Decision Tree via cross-validation
print('Finding best max_depth for Decision Tree...\n')
depth_results = []
for depth in range(2, 21):
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    cv_r2 = cross_val_score(dt, X_train, y_train, cv=5, scoring='r2').mean()
    depth_results.append({'depth': depth, 'cv_r2': cv_r2})

depth_df  = pd.DataFrame(depth_results)
best_depth = depth_df.loc[depth_df['cv_r2'].idxmax(), 'depth']
best_cv_r2 = depth_df.loc[depth_df['cv_r2'].idxmax(), 'cv_r2']
print(f'Best max_depth: {best_depth}  |  Best CV R2: {best_cv_r2:.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(depth_df['depth'], depth_df['cv_r2'], marker='o', linewidth=2, color='#4C72B0')
ax.axvline(best_depth, color='#DD8452', linestyle='--', linewidth=1.5,
           label=f'Best depth = {best_depth}')
ax.set_title('Decision Tree — CV R2 vs Max Depth', fontweight='bold')
ax.set_xlabel('max_depth')
ax.set_ylabel('Cross-validation R2')
ax.legend()
plt.tight_layout()
plt.savefig('/content/chart_07_dt_depth_search.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
dt_model = DecisionTreeRegressor(
    max_depth=best_depth,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
)
dt_results = evaluate_model(dt_model, X_train, X_test, y_train, y_test, 'Decision Tree')

In [ ]:
rf_baseline = RandomForestRegressor(
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
)
rf_results = evaluate_model(rf_baseline, X_train, X_test, y_train, y_test, 'Random Forest (baseline)')

In [ ]:
# Side-by-side comparison table
comparison_data = {
    'Metric': ['MAE (avg $ error)', 'RMSE', 'R2 Test', 'R2 Train', 'R2 5-fold CV', 'Train/test gap'],
    'Decision Tree': [
        f'${dt_results["mae"]:,.0f}', f'${dt_results["rmse"]:,.0f}',
        f'{dt_results["r2_test"]:.4f}', f'{dt_results["r2_train"]:.4f}',
        f'{dt_results["r2_cv"]:.4f}',
        f'{dt_results["r2_train"] - dt_results["r2_test"]:.4f}',
    ],
    'Random Forest': [
        f'${rf_results["mae"]:,.0f}', f'${rf_results["rmse"]:,.0f}',
        f'{rf_results["r2_test"]:.4f}', f'{rf_results["r2_train"]:.4f}',
        f'{rf_results["r2_cv"]:.4f}',
        f'{rf_results["r2_train"] - rf_results["r2_test"]:.4f}',
    ],
}
comparison_df = pd.DataFrame(comparison_data)
print('\nMODEL COMPARISON — ROUND 1')
print('=' * 60)
print(comparison_df.to_string(index=False))
print('=' * 60)

In [ ]:
# Actual vs Predicted for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Actual vs Predicted Salary', fontsize=15, fontweight='bold')
for ax, results, color in zip(axes, [dt_results, rf_results], ['#4C72B0', '#55A868']):
    ax.scatter(y_test, results['y_pred'], alpha=0.4, s=18, color=color, edgecolors='none')
    min_val = min(y_test.min(), results['y_pred'].min())
    max_val = max(y_test.max(), results['y_pred'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_title(f'{results["name"]}\nR2 = {results["r2_test"]:.4f}')
    ax.set_xlabel('Actual Salary (USD)')
    ax.set_ylabel('Predicted Salary (USD)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y,_: f'${y/1000:.0f}k'))
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/content/chart_08_actual_vs_predicted.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Feature importance for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance — What Drives Salary Predictions?',
             fontsize=14, fontweight='bold')
for ax, results, color in zip(axes, [dt_results, rf_results], ['#4C72B0', '#55A868']):
    importances = pd.Series(results['model'].feature_importances_,
                             index=X.columns).sort_values(ascending=True)
    importances.plot(kind='barh', ax=ax, color=color, edgecolor='white')
    ax.set_title(results['name'])
    ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('/content/chart_09_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# GridSearchCV on Random Forest — find best hyperparameters
# This tries every combination and picks the one with highest CV R2
param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [10, 20, None],
    'min_samples_split': [5, 10],
    'min_samples_leaf':  [2, 4],
    'max_features':      ['sqrt', 0.5],
}

print('Running GridSearchCV — takes ~3 minutes...\n')
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    verbose=1,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print('\nBest parameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nBest CV R2: {grid_search.best_score_:.4f}')

In [ ]:
tuned_rf      = grid_search.best_estimator_
tuned_results = evaluate_model(
    tuned_rf, X_train, X_test, y_train, y_test, 'Random Forest (tuned)'
)

In [ ]:
# Before vs After improvement table
print('\n' + '=' * 62)
print('  BEFORE vs AFTER IMPROVEMENTS')
print('=' * 62)
print(f'{"Metric":<28} {"Before":>10} {"After":>10} {"Change":>10}')
print('-' * 62)

rows = [
    ('MAE  (lower=better)',
     f'${rf_results["mae"]:,.0f}', f'${tuned_results["mae"]:,.0f}',
     f'${rf_results["mae"] - tuned_results["mae"]:+,.0f}'),
    ('RMSE (lower=better)',
     f'${rf_results["rmse"]:,.0f}', f'${tuned_results["rmse"]:,.0f}',
     f'${rf_results["rmse"] - tuned_results["rmse"]:+,.0f}'),
    ('R2 test (higher=better)',
     f'{rf_results["r2_test"]:.4f}', f'{tuned_results["r2_test"]:.4f}',
     f'{tuned_results["r2_test"] - rf_results["r2_test"]:+.4f}'),
    ('R2 5-fold CV',
     f'{rf_results["r2_cv"]:.4f}', f'{tuned_results["r2_cv"]:.4f}',
     f'{tuned_results["r2_cv"] - rf_results["r2_cv"]:+.4f}'),
]
for label, bef, aft, chg in rows:
    print(f'  {label:<26} {bef:>10} {aft:>10} {chg:>10}')
print('=' * 62)

In [ ]:
# Save the final tuned model and metadata
FINAL_MODEL_PATH    = '/content/salary_model.pkl'
FINAL_METADATA_PATH = '/content/model_metadata.pkl'

joblib.dump(tuned_rf, FINAL_MODEL_PATH)

model_metadata = {
    'model_type':    'Random Forest (tuned)',
    'feature_names': X.columns.tolist(),
    'best_params':   grid_search.best_params_,
    'metrics': {
        'mae':     round(tuned_results['mae'], 2),
        'rmse':    round(tuned_results['rmse'], 2),
        'r2_test': round(tuned_results['r2_test'], 4),
        'r2_cv':   round(tuned_results['r2_cv'], 4),
    },
}
joblib.dump(model_metadata, FINAL_METADATA_PATH)

print(f'Final model saved    -> {FINAL_MODEL_PATH}')
print(f'Metadata saved       -> {FINAL_METADATA_PATH}')
print(f'Model type           : {model_metadata["model_type"]}')
print(f'R2 test              : {model_metadata["metrics"]["r2_test"]}')
print(f'MAE                  : ${model_metadata["metrics"]["mae"]:,.0f}')

In [ ]:
# Download everything
all_outputs = [
    '/content/salary_model.pkl',
    '/content/model_metadata.pkl',
    '/content/chart_07_dt_depth_search.png',
    '/content/chart_08_actual_vs_predicted.png',
    '/content/chart_09_feature_importance.png',
]
for path in all_outputs:
    files.download(path)
    print(f'Downloaded: {path.split("/")[-1]}')
print('\nPhase 2 complete. All files downloaded.')